# Setup check

Run this once to confirm your environment works. If every cell completes, you are set up.

Before running:
- Add your GitHub personal access token to Colab's Secrets panel (key icon, left sidebar) as `GITHUB_TOKEN`.
- Make sure you have been added to the project's Google Cloud project, so you can read the Earth Engine collection.
- Ask for edit access to the shared Drive folder.

In [ ]:
# Standard header — copy this into every notebook you make
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
TOKEN = userdata.get('GITHUB_TOKEN')
USER = 'JohanEger'   # <-- edit this
PROJECT = 'GEOG761_Team1'
BRANCH = 'main'        # <-- set to your working branch while testing

if not os.path.exists(f'/content/{PROJECT}'):
    !git clone https://{TOKEN}@github.com/{USER}/{PROJECT}.git /content/{PROJECT}
%cd /content/{PROJECT}
!git checkout {BRANCH}
!git pull
!pip install -q -r requirements.txt

import sys; sys.path.append(f'/content/{PROJECT}')

In [ ]:
#DELETE ME
%cd /content/GEOG761_Team1
!git fetch origin
!git checkout initial-setup
!git pull


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Config loads?
from src.config import load_config, data_path
cfg = load_config()
print('CRS:', cfg['crs'])
print('Data root:', cfg['paths']['data_root'])

In [ ]:
# Earth Engine access works?
import ee
ee.Authenticate()
ee.Initialize(project=cfg['gee']['project'])

col = ee.ImageCollection(cfg['gee']['collection'])
n = col.size().getInfo()
print('Scenes in collection:', n)
if n > 0:
    first = ee.Image(col.first())
    print('Bands:', first.bandNames().getInfo())
    print('CRS:', first.select(0).projection().crs().getInfo())
else:
    print('Collection is empty — nothing delivered yet. Band names and CRS '
          'can be checked after the first order.')

In [ ]:
# Shared Drive folder visible? (patches, labels, models, outputs — not imagery)
from pathlib import Path
root = Path(cfg['paths']['data_root'])
print('Exists:', root.exists())
if root.exists():
    print(sorted(p.name for p in root.iterdir()))
else:
    print('Ask to be added to the shared Drive folder, then re-run.')

In [ ]:
# Geospatial stack works?
import rasterio, geopandas, numpy, sklearn, ee
print('rasterio', rasterio.__version__)
print('geopandas', geopandas.__version__)
print('numpy', numpy.__version__)
print('scikit-learn', sklearn.__version__)
print('earthengine-api', ee.__version__)

In [ ]:
# Can you push? Makes an empty commit on a test branch.
!git config user.email "jege510@aucklanduni.ac.nz"   # <-- each person edits
!git config user.name "JohanEger"            # <-- each person edits
!git checkout -b setup-check-$(date +%s)
!git commit --allow-empty -m "Setup check"
!git push origin HEAD
print('If this printed no errors, your token works. Delete the branch on GitHub.')

---
## Saving your work

Copy the cell below into every notebook you make. Colab sessions lose everything in
`/content` when they disconnect, so commit and push before closing the tab.

The `nbstripout` line removes cell outputs before committing. Without it, every run
produces a large diff and notebook merge conflicts become unreadable. It has to run
here rather than as a git filter, because the Colab VM is wiped between sessions.

In [ ]:
# Save your work — run before closing the tab
NAME = 'JohanEger'          # <-- your notebooks/<name>/ folder
BRANCH = 'initial-setup'   # <-- short-lived, one branch per piece of work
MESSAGE = 'Fixed a wrong commit'

!pip install -q nbstripout
!nbstripout notebooks/{NAME}/*.ipynb

!git checkout -b {BRANCH} 2>/dev/null || git checkout {BRANCH}
!git add notebooks/{NAME}/ src/ docs/
!git commit -m "{MESSAGE}"
!git push -u origin {BRANCH}

In [ ]:
# Save your work — run before closing the tab
import json, os
from google.colab import _message

NAME = 'johan'                    # <-- your notebooks/<name>/ folder
NOTEBOOK = 'explore_scenes.ipynb' # <-- filename to save as
BRANCH = 'johan/explore'          # <-- short-lived, one branch per piece of work
MESSAGE = 'Describe what you did'
EMAIL, AUTHOR = 'you@example.com', 'Your Name'   # <-- your GitHub identity

# Pull the live notebook out of the Colab session
nb = _message.blocking_request('get_ipynb', timeout_sec=60)['ipynb']

# Strip outputs before writing
for cell in nb['cells']:
    cell['outputs'] = [] if cell['cell_type'] == 'code' else cell.get('outputs', [])
    cell['execution_count'] = None if cell['cell_type'] == 'code' else None

os.makedirs(f'notebooks/{NAME}', exist_ok=True)
with open(f'notebooks/{NAME}/{NOTEBOOK}', 'w') as f:
    json.dump(nb, f, indent=1)

!git config user.email "{EMAIL}"
!git config user.name "{AUTHOR}"
!git checkout -b {BRANCH} 2>/dev/null || git checkout {BRANCH}
!git add notebooks/{NAME}/ src/ docs/
!git commit -m "{MESSAGE}"
!git push -u origin {BRANCH}